<a href="https://colab.research.google.com/github/viniciusarnom/estudo_dirigido/blob/main/Random%20Forest%20Classifica%C3%A7%C3%A3o%20de%20Label%20-%201000.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# 1. Carregar os quatro arquivos CSV de features
print("Carregando arquivos...")
df_bes = pd.read_pickle("concatenated_features_bes_day1.bin").head(1000)
df_browning = pd.read_pickle("concatenated_features_browning_day1.bin").head(1000)
df_honors = pd.read_pickle("concatenated_features_honors_day1.bin").head(1000)
df_meb = pd.read_pickle("concatenated_features_meb_day1.bin").head(1000)

# 2. Atribuir uma Label única para cada localização/roteador
# Isso garante que o modelo saiba diferenciar as 4 origens
df_bes['Label'] = 0
df_browning['Label'] = 1
df_honors['Label'] = 2
df_meb['Label'] = 3

# 3. Concatenar todos os DataFrames em um único conjunto de dados
print("Concatenando os dados...")
df_final = pd.concat([df_bes, df_browning, df_honors, df_meb], ignore_index=True)

# 4. Preparar as Features (X) e o Alvo/Target (y)
X = df_final[['I', 'Q', 'Magnitude', 'Spectrum']]
y = df_final['Label']

# 5. Dividir os dados em Treino (70%) e Teste (30%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 6. Criar e treinar o modelo Random Forest
print("Treinando o modelo Random Forest Multiclasse (Isso pode levar alguns segundos)...")
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_clf.fit(X_train, y_train)

# 7. Fazer as previsões usando os dados de teste invisíveis ao modelo
print("Calculando previsões e métricas...")
y_pred = rf_clf.predict(X_test)

# 8. Calcular as métricas
# Usamos average='weighted' para calcular a média ponderada entre as 4 classes
acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

# 9. Exibir os resultados globais
print("\n--- Resultados Globais da Classificação (4 Classes) ---")
print(f"Acurácia (Accuracy): {acc:.4f}")
print(f"Precisão (Precision): {precision:.4f}")
print(f"Recall:              {recall:.4f}")
print(f"F1-Score:            {f1:.4f}")

# 10. Exibir o relatório detalhado para ver o desempenho em CADA roteador separadamente
print("\n--- Relatório Detalhado por Roteador ---")
nomes_classes = ['bes (0)', 'browning (1)', 'honors (2)', 'meb (3)']
print(classification_report(y_test, y_pred, target_names=nomes_classes, zero_division=0))

Carregando arquivos...
Concatenando os dados...
Treinando o modelo Random Forest Multiclasse (Isso pode levar alguns segundos)...
Calculando previsões e métricas...

--- Resultados Globais da Classificação (4 Classes) ---
Acurácia (Accuracy): 0.4875
Precisão (Precision): 0.4814
Recall:              0.4875
F1-Score:            0.4797

--- Relatório Detalhado por Roteador ---
              precision    recall  f1-score   support

     bes (0)       0.36      0.34      0.35       315
browning (1)       0.72      0.65      0.68       307
  honors (2)       0.33      0.27      0.30       294
     meb (3)       0.52      0.71      0.60       284

    accuracy                           0.49      1200
   macro avg       0.48      0.49      0.48      1200
weighted avg       0.48      0.49      0.48      1200



In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier # Importação da Rede Neural
from sklearn.preprocessing import StandardScaler # Essencial para Redes Neurais
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# 1. Carregar os arquivos (Mantido conforme original)
print("Carregando arquivos...")
df_bes = pd.read_pickle("concatenated_features_bes_day1.bin").head(1000)
df_browning = pd.read_pickle("concatenated_features_browning_day1.bin").head(1000)
df_honors = pd.read_pickle("concatenated_features_honors_day1.bin").head(1000)
df_meb = pd.read_pickle("concatenated_features_meb_day1.bin").head(1000)

# 2. Atribuir Labels
df_bes['Label'] = 0
df_browning['Label'] = 1
df_honors['Label'] = 2
df_meb['Label'] = 3

# 3. Concatenar
print("Concatenando os dados...")
df_final = pd.concat([df_bes, df_browning, df_honors, df_meb], ignore_index=True)

# 4. Preparar Features (X) e Target (y)
X = df_final[['I', 'Q', 'Magnitude', 'Spectrum']]
y = df_final['Label']

# 5. Dividir os dados
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# --- NOVIDADE: Normalização ---
# Redes neurais funcionam muito melhor quando os dados estão na mesma escala (média 0, desvio 1)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 6. Criar e treinar o modelo MLP (Rede Neural)
print("Treinando a Rede Neural MLP (Isso pode levar alguns segundos)...")
# Configuração: 2 camadas escondidas de 100 e 50 neurônios, respectivamente.
mlp_clf = MLPClassifier(
    hidden_layer_sizes=(100, 50),
    max_iter=500,
    activation='relu',
    solver='adam',
    random_state=42
)
mlp_clf.fit(X_train, y_train)

# 7. Fazer as previsões
print("Calculando previsões e métricas...")
y_pred = mlp_clf.predict(X_test)

# 8. Calcular as métricas (Mantido igual)
acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

# 9. Exibir os resultados globais
print("\n--- Resultados Globais da Classificação (4 Classes) ---")
print(f"Acurácia (Accuracy): {acc:.4f}")
print(f"Precisão (Precision): {precision:.4f}")
print(f"Recall:              {recall:.4f}")
print(f"F1-Score:            {f1:.4f}")

print("\n--- Relatório Detalhado por Roteador ---")
nomes_classes = ['bes (0)', 'browning (1)', 'honors (2)', 'meb (3)']
print(classification_report(y_test, y_pred, target_names=nomes_classes, zero_division=0))

Carregando arquivos...
Concatenando os dados...
Treinando a Rede Neural MLP (Isso pode levar alguns segundos)...
Calculando previsões e métricas...

--- Resultados Globais da Classificação (4 Classes) ---
Acurácia (Accuracy): 0.5275
Precisão (Precision): 0.5323
Recall:              0.5275
F1-Score:            0.5187

--- Relatório Detalhado por Roteador ---
              precision    recall  f1-score   support

     bes (0)       0.40      0.35      0.37       315
browning (1)       0.80      0.65      0.72       307
  honors (2)       0.41      0.32      0.36       294
     meb (3)       0.52      0.80      0.63       284

    accuracy                           0.53      1200
   macro avg       0.53      0.53      0.52      1200
weighted avg       0.53      0.53      0.52      1200



In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Carregar os arquivos (Mantido conforme original)
print("Carregando arquivos...")
df_bes = pd.read_pickle("concatenated_features_bes_day1.bin").head(1000)
df_browning = pd.read_pickle("concatenated_features_browning_day1.bin").head(1000)
df_honors = pd.read_pickle("concatenated_features_honors_day1.bin").head(1000)
df_meb = pd.read_pickle("concatenated_features_meb_day1.bin").head(1000)

# 2. Atribuir Labels
df_bes['Label'] = 0
df_browning['Label'] = 1
df_honors['Label'] = 2
df_meb['Label'] = 3

# 3. Concatenar
print("Concatenando os dados...")
df_final = pd.concat([df_bes, df_browning, df_honors, df_meb], ignore_index=True)

# 4. Preparar Features (X) e Target (y)
X = df_final[['I', 'Q', 'Magnitude', 'Spectrum']].values
y = df_final['Label'].values

# 5. Dividir os dados
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# --- AJUSTES PARA CNN ---

# A. Normalização (CNNs são sensíveis à escala)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# B. Reshape: A CNN 1D espera (amostras, passos_no_tempo, features)
# Como temos 4 colunas independentes por linha, faremos (N, 4, 1)
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

# 6. Criar a Arquitetura da CNN
model = models.Sequential([
    # Camada Convolucional: Extrai padrões locais entre as features
    layers.Conv1D(filters=32, kernel_size=2, activation='relu', input_shape=(4, 1)),
    layers.MaxPooling1D(pool_size=2),
    layers.Flatten(),
    # Camada Densa de classificação
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2), # Ajuda a evitar overfitting
    layers.Dense(4, activation='softmax') # 4 classes de saída
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# 7. Treinar o modelo
print("Treinando a CNN...")
model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.1, verbose=1)

# 8. Fazer as previsões
print("Calculando previsões...")
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1) # Converte probabilidades para a classe com maior valor

# 9. Exibir Resultados
acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

# 10. Exibir os resultados globais
print("\n--- Resultados Globais da Classificação (4 Classes) ---")
print(f"Acurácia (Accuracy): {acc:.4f}")
print(f"Precisão (Precision): {precision:.4f}")
print(f"Recall:              {recall:.4f}")
print(f"F1-Score:            {f1:.4f}")

# 10. Relatório Detalhado
print("\n--- Relatório Detalhado por Roteador ---")
nomes_classes = ['bes (0)', 'browning (1)', 'honors (2)', 'meb (3)']
print(classification_report(y_test, y_pred, target_names=nomes_classes, zero_division=0))

Carregando arquivos...
Concatenando os dados...
Treinando a CNN...
Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.3873 - loss: 1.2774 - val_accuracy: 0.4929 - val_loss: 1.1715
Epoch 2/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5016 - loss: 1.1213 - val_accuracy: 0.5571 - val_loss: 1.0681
Epoch 3/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5222 - loss: 1.0565 - val_accuracy: 0.5679 - val_loss: 1.0331
Epoch 4/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5254 - loss: 1.0289 - val_accuracy: 0.5786 - val_loss: 1.0288
Epoch 5/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5310 - loss: 1.0247 - val_accuracy: 0.5714 - val_loss: 1.0248
Epoch 6/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5369 - loss: 1.0161 - val_accuracy: 0.5571 - val_loss: 1.0225
Epoch 7/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5317 - loss: 1.0084 - val_accuracy: 0.5643 - val_loss: 1.0254
Epoch 8/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5222 - loss: 1.0106 - val_accuracy: 0.5893 - val_loss: 1.0190
Epo

In [15]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn import metrics

# 1. Carregar os arquivos (Mantido)
print("Carregando arquivos...")
df_bes = pd.read_pickle("concatenated_features_bes_day1.bin").head(1000)
df_browning = pd.read_pickle("concatenated_features_browning_day1.bin").head(1000)
df_honors = pd.read_pickle("concatenated_features_honors_day1.bin").head(1000)
df_meb = pd.read_pickle("concatenated_features_meb_day1.bin").head(1000)

# 2. Atribuir Labels (Apenas para comparação posterior, o DBSCAN não as usa no treino)
df_bes['Label'] = 0
df_browning['Label'] = 1
df_honors['Label'] = 2
df_meb['Label'] = 3

# 3. Concatenar
print("Concatenando os dados...")
df_final = pd.concat([df_bes, df_browning, df_honors, df_meb], ignore_index=True)

# 4. Preparar Features (X) e o Alvo real (y) para validação
X = df_final[['I', 'Q', 'Magnitude', 'Spectrum']]
y_true = df_final['Label']

# --- AJUSTES PARA DBSCAN ---

# A. Normalização: OBRIGATÓRIA para DBSCAN, pois ele se baseia em distância Euclidiana
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 6. Criar e "treinar" o modelo DBSCAN
# eps: Distância máxima entre dois pontos para serem considerados vizinhos
# min_samples: Mínimo de pontos para formar uma região densa (cluster)
print("Rodando DBSCAN para encontrar clusters...")
dbscan = DBSCAN(eps=0.5, min_samples=10)
clusters = dbscan.fit_predict(X_scaled)

# 7. Analisar os resultados
# O DBSCAN retorna -1 para pontos que considera ruído (outliers)
n_clusters_ = len(set(clusters)) - (1 if -1 in clusters else 0)
n_noise_ = list(clusters).count(-1)

# 8. Calcular métricas intrínsecas (já que não há "previsão" de classes rotuladas)
print("\n--- Resultados do Agrupamento (DBSCAN) ---")
print(f"Número estimado de clusters: {n_clusters_}")
print(f"Número estimado de pontos de ruído: {n_noise_} de {len(X)}")

if n_clusters_ > 1:
    print(f"Coeficiente de Silhueta: {metrics.silhouette_score(X_scaled, clusters):.3f}")

# 9. Comparação com as Labels Originais (Ajustado)
# Nota: O DBSCAN pode encontrar 2 clusters ou 50, dependendo dos parâmetros.
print("\n--- Distribuição de Clusters encontrados vs Labels Originais ---")
df_resultado = pd.DataFrame({'Label_Real': y_true, 'Cluster_DBSCAN': clusters})
print(pd.crosstab(df_resultado['Label_Real'], df_resultado['Cluster_DBSCAN']))

Carregando arquivos...
Concatenando os dados...
Rodando DBSCAN para encontrar clusters...

--- Resultados do Agrupamento (DBSCAN) ---
Número estimado de clusters: 3
Número estimado de pontos de ruído: 722 de 4000
Coeficiente de Silhueta: 0.327

--- Distribuição de Clusters encontrados vs Labels Originais ---
Cluster_DBSCAN   -1    0   1   2
Label_Real                      
0                85  915   0   0
1               575  402  14   9
2                50  950   0   0
3                12  988   0   0
